# ECG Insight AI — Exploratory Data Analysis & Preprocessing
This notebook demonstrates biomedical time-series data loading, zero-phase filtering, Pan-Tompkins QRS peak detection, and feature extraction on the **PhysioNet MIT-BIH Arrhythmia Database**.

In [ ]:
import sys
import os
from pathlib import Path
# Ensure workspace root is in sys.path when running from notebooks directory
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_mitdb_record, generate_synthetic_ecg
from src.preprocessing import preprocess_ecg_signal, pan_tompkins_r_peak_detection
from src.features import segment_heartbeats, extract_ecg_features

print('All biomedical modules loaded successfully!')


## 1. Load PhysioNet MIT-BIH Record 106 (Frequent PVCs)

In [ ]:
rec = load_mitdb_record('106', max_seconds=30)
signal = rec['signal']
fs = rec['fs']
time = np.arange(len(signal)) / fs
print(f'Loaded {rec["record_name"]}: {len(signal)} samples, fs={fs} Hz, duration={len(signal)/fs}s')

## 2. Digital Signal Processing (Zero-Phase Butterworth & Notch Filtering)

In [ ]:
prep = preprocess_ecg_signal(signal, fs=fs, lowcut=0.5, highcut=40.0, apply_notch=True)
cleaned = prep['cleaned']

plt.figure(figsize=(14, 5))
plt.plot(time[:1500], signal[:1500], label='Raw ECG (with drift & noise)', color='#ef4444', alpha=0.7)
plt.plot(time[:1500], cleaned[:1500], label='Cleaned ECG (0.5-40Hz Butterworth)', color='#06b6d4', linewidth=1.5)
plt.title('Raw vs. Zero-Phase Filtered ECG Signal')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (mV)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Pan-Tompkins QRS Detection & Heartbeat Segmentation

In [ ]:
r_peaks = pan_tompkins_r_peak_detection(cleaned, fs=fs)
valid_peaks, waveforms = segment_heartbeats(cleaned, r_peaks, fs=fs)
features, feature_names = extract_ecg_features(waveforms, valid_peaks, fs=fs)

print(f'Detected {len(r_peaks)} R-peaks.')
print(f'Extracted {features.shape[0]} segmented beats with {features.shape[1]} features each.')